In [1]:
# --- Instalación de librerías si no las tenés ---
# !pip install pytesseract pillow pandas

import pytesseract
from PIL import Image
import pandas as pd
import re

# --------------------------------------------------
# 1) INDICAR RUTA DEL ARCHIVO PNG
# --------------------------------------------------

image_path = "Remittance_Cruz_Verde.png"  # <-- Cambiar si corresponde
img = Image.open(image_path)

# --------------------------------------------------
# 2) EXTRAER TEXTO CON OCR
# --------------------------------------------------

raw_text = pytesseract.image_to_string(img, lang="spa")

# --------------------------------------------------
# 3) PROCESAR TEXTO PARA EXTRAER TABLA
# --------------------------------------------------

lines = raw_text.split("\n")

facturas = []
importes = []

for line in lines:
    # Coincide con número de factura
    match_fact = re.search(r"\b(PMP\d+|NCMI\d+)\b", line)
    # Coincide con importe: puede tener comas y paréntesis
    match_imp = re.search(r"[\(]?\d{1,3}(?:,\d{3})*(?:\))?", line)

    if match_fact and match_imp:
        factura = match_fact.group(0)
        importe_raw = match_imp.group(0)

        facturas.append(factura)
        importes.append(importe_raw)

# --------------------------------------------------
# 4) LIMPIEZA DEL CAMPO "Importe Pagado"
# --------------------------------------------------

def limpiar_importe(x):
    x = x.replace(",", "")
    if x.startswith("(") and x.endswith(")"):
        return -int(x.strip("()"))
    return int(x)

importes_limpios = [limpiar_importe(i) for i in importes]

# --------------------------------------------------
# 5) CREAR DATAFRAME FINAL
# --------------------------------------------------

df = pd.DataFrame({
    "numero_factura": facturas,
    "importe_pagado": importes_limpios
})

df


TesseractError: (1, 'Error opening data file /usr/local/share/tessdata/spa.traineddata Please make sure the TESSDATA_PREFIX environment variable is set to your "tessdata" directory. Failed loading language \'spa\' Tesseract couldn\'t load any languages! Could not initialize tesseract.')